In [9]:
import pandas as pd

Вы работаете аналитиком в сети супермакетов, и ваша задача - автоматизировать работу с торговыми точками.\
Ваша задача - исследовать различные торговые точки на предмет их отклонения от среднего чека.\
Если средний чек в торговой точке меньше, чем средний чек по всем наблюдениям, умноженный на определенное пороговое значение, то торговая точка помечается (как Булево значение False).\
Также необходимо работать только с теми торговыми точками, в которых достаточное количество наблюдений, чтобы "верить" среднему чеку.

### Задание 1

Импортируйте файл **'Лямбда-функции. Список транзакций.xlsx'** \
Рассчитайте среднее значение чека по всем операциям в таблице. \
Рассчитайте среднее значение чека и количество транзакций для каждой торговой точки по отдельности.


In [10]:
df = pd.read_excel('Лямбда-функции. Список транзакций.xlsx')

avg_all = df['check'].mean()
print(f"Средний чек по всем операциям: {avg_all:.2f}")

grouped = df.groupby('id_point').agg(
    avg_check=('check', 'mean'),
    cnt_trans=('check', 'count')
).reset_index()

print(grouped)

Средний чек по всем операциям: 3930.85
   id_point    avg_check  cnt_trans
0       101  4038.350852        704
1       102  5335.203008        532
2       103  3857.566596        473
3       104  3893.656000        375
4       105  3883.217064        797
5       106  5065.804348        736
6       107  2936.442130        432
7       108  3025.939058        361
8       109  2098.698876        445


### Задание 2

Создайте функцию, которая на вход будет брать следующие аргументы:\
\
    1. Средний чек, который мы проверяем \
    2. Кол-во транзакций, которое мы проверяем \
    3. Пороговое значение среднего чека \
    4. Пороговое значение кол-ва транзакций \
    5. Процент отступления от порогового значения среднего чека (со значением по умолчанию 0.9)

Если количество наблюдений меньше чем пороговое значение количества наблюдений, то функция принтит "Мало наблюдений!".

Если проверяемый средний чек меньше, чем пороговый средний чек, умноженный на "процент отступления", то функция возвращает False.

В противном случае функция возвращает True.

Также с помощью выражения except обработайте ошибку типизации данных (TypeError), выведите сообщение "Нужны цифры!".

In [11]:
def check_point(avg_check, cnt_trans, avg_thr, cnt_thr, delta=0.9):
    try:
        if cnt_trans < cnt_thr:
            print("Мало наблюдений!")
            return None
        
        if avg_check < avg_thr * delta:
            return False
        else:
            return True
    except TypeError:
        print("Нужны цифры!")
        return None

Протестируйте функцию на параметрах торговой точки 101 (4038, 704).\
В качестве порогового значения количества наблюдений возьмите 300.\
В качестве порогового значения среднего чека возьмите рассчитанный в прошлом пункте средний чек по всем торговым точкам вместе.

In [12]:
cnt_thr = 300

In [13]:
result = check_point(
    avg_check=4038,
    cnt_trans=704,
    avg_thr=avg_all,
    cnt_thr=cnt_thr
)
print(f"Результат проверки точки 101: {result}")

Результат проверки точки 101: True


### Задание 3

С помощью операции маппинга (mapping) примените данную функцию ко всем торговым точкам из датасета.\
Для этого создайте в сгруппированном датафрейме две дополнительные колонки: одну заполните пороговым значением среднего чека, а другую - пороговым значением количества наблюдений.\
После этого можно передать 4 колонки сгруппированного датафрейма в качестве аргументов функции map, наравне с вашей функций.\
Сохраните результат в список.\
Рассчитайте долю прошедших проверку (True) торговых точек. Для этого разделите сумму значений списка на его длину.

In [14]:
grouped['avg_thr'] = avg_all
grouped['cnt_thr'] = cnt_thr

results = list(map(
    check_point,
    grouped['avg_check'],
    grouped['cnt_trans'],
    grouped['avg_thr'],
    grouped['cnt_thr']
))

grouped['check_result'] = results
print(grouped.head())

share_true = sum(1 for x in results if x is True) / len(results)
print(f"Доля прошедших проверку точек: {share_true:.4f}")

   id_point    avg_check  cnt_trans      avg_thr  cnt_thr  check_result
0       101  4038.350852        704  3930.846756      300          True
1       102  5335.203008        532  3930.846756      300          True
2       103  3857.566596        473  3930.846756      300          True
3       104  3893.656000        375  3930.846756      300          True
4       105  3883.217064        797  3930.846756      300          True
Доля прошедших проверку точек: 0.6667
